In [ ]:
# Importing necessary libraries for data processing, clustering, and visualization
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns

# Setting random seed for reproducibility
np.random.seed(42)

# Loading the dataset
df = pd.read_excel("ready_for_modeling_1.xlsx")

# --- Creating reasonClosure Column ---
# Mapping reasons to indices: 0=Financial, 1=Enrollment, 2=Pandemic, 3=MutualBenefit
reason_cols = ['reasonFinancial', 'reasonEnrollment', 'reasonPandemic', 'reasonMutualBenefit']
def create_reason_closure(row):
    reasons = [str(i) for i, reason in enumerate(reason_cols) if row[reason]]
    return ''.join(sorted(reasons)) if reasons else 'None'

df['reasonClosure'] = df.apply(create_reason_closure, axis=1)

# --- Data Preprocessing ---
# Keeping name as label but excluding from clustering
labels = df[['name', 'reasonClosure']].copy()
X_df = df.drop(columns=['name', 'reasonClosure'] + reason_cols)

# Imputing missing endowmentMedian with -1 to indicate no data
X_df['endowmentMedian'] = X_df['endowmentMedian'].fillna(-1)

# Identifying cat and num columns
cat_cols = X_df.select_dtypes(include=['object', 'bool']).columns.tolist()
num_cols = X_df.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Defining preprocessing for num data: scale
num_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Defining preprocessing for cat data: one-hot encode
cat_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

# Combining preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

# Fitting and transforming the data
X = preprocessor.fit_transform(X_df)

# Getting feature names after one-hot encoding
cat_feature_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(cat_cols)
feature_names = np.concatenate([num_cols, cat_feature_names])
X_processed = pd.DataFrame(X, columns=feature_names)